In [11]:
import os
import pandas as pd
from IPython.display import display
from tqdm import tqdm
import re

folder = os.getcwd()
excels = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".xlsx")]

In [12]:
dfs = []
for i, pth in enumerate(tqdm(excels)):
    df_i = pd.read_excel(pth)
    df_i = df_i.loc[:, ['Part Number', 'Models']]
    dfs.append(df_i)

df_all = pd.concat(dfs, ignore_index=True)

100%|██████████| 20/20 [00:01<00:00, 15.06it/s]


In [13]:
def join_ordered(series: pd.Series):
    all_parts = []

    for item in series.dropna(): 
        parts = str(item).split(';') 
        for p in parts:
            p_clean = re.sub(r'\s+', '', p).upper()
            if p_clean:
                all_parts.append(p_clean)

    seen = set()
    result = []
    for v in all_parts:
        if v not in seen:
            seen.add(v)
            result.append(v)

    return ";".join(result) if result else None


In [14]:
def merge_parts(df):
    df = df.copy()

    agg_map = {
        "Models": join_ordered
    }

    df_agg = df.groupby("Part Number", as_index=False).agg(agg_map)

    return df_agg

In [15]:
df_all.info()

df_all["Part Number"] = df_all["Part Number"].astype(str).str.strip()

part_numeric = pd.to_numeric(df_all["Part Number"], errors="coerce")

df_letters = df_all[part_numeric.isna()].copy()
df_numbers = df_all[part_numeric.notna()].copy()

if not df_numbers.empty:
    df_numbers["Part Number"] = (
        part_numeric[part_numeric.notna()]
        .astype(int)
        .astype(str)
        .str.zfill(8)
    )

<class 'pandas.DataFrame'>
RangeIndex: 22949 entries, 0 to 22948
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Part Number  22949 non-null  object
 1   Models       22949 non-null  str   
dtypes: object(1), str(1)
memory usage: 358.7+ KB


In [16]:
df_numbers_merged = merge_parts(df_numbers) if not df_numbers.empty else df_numbers
df_letters_merged = merge_parts(df_letters) if not df_letters.empty else df_letters

df_merged = pd.concat([df_letters_merged, df_numbers_merged], ignore_index=True)

In [18]:
df_merged.to_excel(r"final_модели.xlsx", sheet_name="Parts", index=False)